In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt
from scipy import signal

KeyboardInterrupt: 

In [ ]:
def passa_baixa(m, fc_hertz, fs):
    h = np.zeros(m)
    pi = 3.14159265
    fc = fc_hertz/fs #0.125 -> nao pode passar de 0.5

    for i in range(m):
        if (i - m/2) == 0: #ponto central -> divisão por 0
            h[i] = 2 * pi * fc
        else:
            h[i] = math.sin(2 * pi * fc * (i - m/2)) / (i - m/2)
        # Janela de Hamming
        h[i] = h[i] * (0.54 - 0.46 * math.cos(2 * pi * i / m))

    #soma todos os coeficientes
    soma = 0
    for i in range(m):
        soma = soma + h[i]

    for i in range(m):
        h[i] = h[i] / soma
    
    return h

: 

In [ ]:
def ler_arquivo_hex(arquivo):
    valores = []
    
    with open(arquivo, 'r') as arquivo:
        for linha in arquivo:
            linha = linha.strip() # Remove espaços e quebras de linha
            if not linha:
                continue
            valor_inteiro = int(linha, 16) #le em hexa e converte pra int
            #se tiver numeros negativos
            if valor_inteiro >= 32768:
                valor_inteiro -= 65536

            valores.append(valor_inteiro)

    vetor_x = np.array(valores) / 32767.0 #converte p array do numpy e volta pra -1 a 1

    return vetor_x

In [ ]:
def lms(w, x, e):
    mu = 10**-6
    n = len(w) 

    for i in range (n, 0, -1):
        w[i] = w[i] + mu * e[i] * x[n-i]
        
    return w

In [ ]:
def filtro(w, x):
    N = len(w)
    soma = 0
    for j in range(N):
        soma = soma + w[j] * x[j]
    
    return soma 

In [ ]:
m = 50
fc_hertz = 1000
fs = 8000

X = ler_arquivo_hex('ruido_branco.hex')
T = passa_baixa(m, fc_hertz, fs)

E = np.zeros(len(X))
W = np.zeros(m)

janela_x = np.zeros(m)

In [ ]:
for i in range(len(X)):
    for j in range(m-1, 0, -1):
        janela_x[j] = janela_x[j-1]
    janela_x[0] = X[i]

    D = filtro(T, janela_x)
    Y = filtro(W, janela_x)

    e = D - Y
    E[i] = e

    W = lms(W, janela_x, e)

In [ ]:
plt.plot(E)